In [1]:
import numpy as np
import pandas as pd
import gower
import pickle
import logging
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
import kmedoids
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import umap
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Importing distance functions
import sys
sys.path.append(str(Path.cwd()))
from distance import mixed_matrix_distance

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

# ── Load data ────────────────────────────────────────────────────────────────
df_metadata = pd.read_excel("C:/Users/giuli/Repositories/fintech-group-work/BusinessCase1/Data/BankClients_Metadata.xlsx")
print(f"Loaded: {df_metadata.shape[0]:,} rows × {df_metadata.shape[1]} columns")
df_metadata.head()

c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
df_raw   = pd.read_excel(r"C:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\Data\Dataset1_BankClients.xlsx")
print(f"Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head()


Loaded: 5,000 rows × 18 columns


,ID,Age,Gender,Job,Area,CitySize,FamilySize,Income,Wealth,Debt,FinEdu,ESG,Digital,BankFriend,LifeStyle,Luxury,Saving,Investments
0,1,24,1,1,2,2,4,0.668046,0.702786,0.262070,0.741853,0.483684,0.698625,0.618259,0.607877,0.897369,0.283222,1
1,2,47,1,2,2,3,1,0.858453,0.915043,0.730430,0.859423,0.537167,0.959025,0.785936,0.862271,0.913729,0.821590,3
2,3,38,0,2,1,2,2,0.926818,0.898316,0.441272,0.485953,0.649434,0.750265,0.699725,0.755404,0.765199,0.503790,3
3,4,67,0,2,1,2,3,0.538797,0.423180,0.600401,0.493144,0.533829,0.590165,0.675353,0.334432,0.517209,0.691240,2
4,5,33,0,2,1,3,1,0.806659,0.731404,0.831449,0.856286,0.784940,0.710026,0.758793,0.908878,0.611610,0.615916,2


In [ ]:
# ── Column definitions ───────────────────────────────────────────────────────
categorical_cols = ["Gender", "Job", "Area"]
numerical_cols   = [c for c in df_raw.columns
                    if c not in categorical_cols + ["ID"]]

# ── 1. Drop ID ───────────────────────────────────────────────────────────────
df = df_raw.drop(columns=["ID"])

# ── 2. Missing values ────────────────────────────────────────────────────────
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
print(f"Missing values after imputation: {df.isna().sum().sum()}")

# ── 3. Hard domain filter — working minors ───────────────────────────────────
mask_minors = (df["Age"] < 18) & (df["Job"].isin([2, 3, 4]))
df = df[~mask_minors].reset_index(drop=True)
print(f"After minor filter: {len(df):,} rows")

# ── 4. Isolation Forest — remove top 1% multivariate outliers ────────────────
iso    = IsolationForest(contamination=0.01, random_state=42)
labels = iso.fit_predict(df[numerical_cols])
df     = df[labels == 1].reset_index(drop=True)
print(f"After outlier removal: {len(df):,} rows")

# ── 5. Prepare input arrays ──────────────────────────────────────────────────
# Numerical: min-max scale
scaler = MinMaxScaler()
X_num  = scaler.fit_transform(df[numerical_cols])

# Categorical: label-encode (for Hamming)
X_cat_int = df[categorical_cols].apply(
    lambda col: LabelEncoder().fit_transform(col)
).values

# Categorical: one-hot encode (for Tanimoto) — astype(str) forces proper encoding
X_cat_ohe = pd.get_dummies(
    df[categorical_cols].astype(str)
).values.astype(np.int32)

# Gower: needs original df with categorical cols as strings
df_gower = df.copy()
for col in categorical_cols:
    df_gower[col] = df_gower[col].astype(str)

print(f"\nX_num shape:     {X_num.shape}")
print(f"X_cat_int shape: {X_cat_int.shape}  — unique: {np.unique(X_cat_int)}")
print(f"X_cat_ohe shape: {X_cat_ohe.shape}  — unique: {np.unique(X_cat_ohe)}")

Missing values after imputation: 0
After minor filter: 5,000 rows
After outlier removal: 4,950 rows

X_num shape:     (4950, 14)
X_cat_int shape: (4950, 3)  — unique: [0 1 2 3 4]
X_cat_ohe shape: (4950, 10)  — unique: [0 1]


In [ ]:
mixed_configs = [
    ("L1",       "Hamming",  X_cat_int.copy()),
    ("L1",       "Tanimoto", X_cat_ohe.copy()),
    ("L2",       "Hamming",  X_cat_int.copy()),
    ("L2",       "Tanimoto", X_cat_ohe.copy()),
    ("Canberra", "Hamming",  X_cat_int.copy()),
    ("Canberra", "Tanimoto", X_cat_ohe.copy()),
]

# Sanity check
for num_dist, cat_dist, X_cat in mixed_configs:
    print(f"{num_dist}+{cat_dist}: dtype={X_cat.dtype}, unique={np.unique(X_cat)}, shape={X_cat.shape}")

L1+Hamming: dtype=int64, unique=[0 1 2 3 4], shape=(4950, 3)
L1+Tanimoto: dtype=int32, unique=[0 1], shape=(4950, 10)
L2+Hamming: dtype=int64, unique=[0 1 2 3 4], shape=(4950, 3)
L2+Tanimoto: dtype=int32, unique=[0 1], shape=(4950, 10)
Canberra+Hamming: dtype=int64, unique=[0 1 2 3 4], shape=(4950, 3)
Canberra+Tanimoto: dtype=int32, unique=[0 1], shape=(4950, 10)


In [ ]:
import pickle
from pathlib import Path

CACHE_DIR = Path("distance_cache")
CACHE_DIR.mkdir(exist_ok=True)

def load_or_compute(path: Path, compute_fn):
    """Load matrix from pickle if it exists, otherwise compute and save."""
    if path.exists():
        logger.info("Loading cached matrix from %s", path)
        with open(path, "rb") as f:
            return pickle.load(f)
    logger.info("Computing matrix — will save to %s", path)
    D = compute_fn()
    with open(path, "wb") as f:
        pickle.dump(D, f)
    logger.info("Saved to %s", path)
    return D

# ── Mixed: all combinations ────────────────────────────────────────────────
mixed_matrices = {}
for num_dist, cat_dist, X_cat in mixed_configs:
    key      = f"Mixed ({num_dist}+{cat_dist})"
    pkl_name = f"D_mixed_{num_dist.lower()}_{cat_dist.lower()}.pkl"
    mixed_matrices[key] = load_or_compute(
        CACHE_DIR / pkl_name,
        lambda nd=num_dist, cd=cat_dist, xc=X_cat: mixed_matrix_distance(
            X_num, xc, alpha=0.5,
            num_dist=nd, cat_dist=cd
        )
    )

# ── Gower ──────────────────────────────────────────────────────────────────
D_gower = load_or_compute(
    CACHE_DIR / "D_gower.pkl",
    lambda: gower.gower_matrix(df_gower)
)

print("\nAll matrices ready:")
for key in mixed_matrices:
    print(f"  {key}: {mixed_matrices[key].shape}")
print(f"  Gower: {D_gower.shape}")

2026-03-17 21:28:08,241 — INFO — Computing matrix — will save to distance_cache\D_mixed_l1_hamming.pkl
2026-03-17 21:28:08,242 — INFO — Computing mixed distance matrix (L1 + Hamming, alpha=0.50) for 4950 samples using -1 jobs.
2026-03-17 21:31:19,980 — INFO — Mixed distance matrix computed. Shape: (4950, 4950)
2026-03-17 21:31:20,494 — INFO — Saved to distance_cache\D_mixed_l1_hamming.pkl
2026-03-17 21:31:20,495 — INFO — Computing matrix — will save to distance_cache\D_mixed_l1_tanimoto.pkl
2026-03-17 21:31:20,496 — INFO — Computing mixed distance matrix (L1 + Tanimoto, alpha=0.50) for 4950 samples using -1 jobs.
2026-03-17 21:46:37,483 — INFO — Mixed distance matrix computed. Shape: (4950, 4950)
2026-03-17 21:46:38,028 — INFO — Saved to distance_cache\D_mixed_l1_tanimoto.pkl
2026-03-17 21:46:38,032 — INFO — Computing matrix — will save to distance_cache\D_mixed_l2_hamming.pkl
2026-03-17 21:46:38,037 — INFO — Computing mixed distance matrix (L2 + Hamming, alpha=0.50) for 4950 samples u

TypeError: Cannot interpret '<StringDtype(na_value=nan)>' as a data type

In [ ]:
import kmedoids

K_RANGE  = range(2, 11)
matrices = {

    **mixed_matrices,
    #"Gower":    D_gower,
}

results = {}

for name, D in matrices.items():
    logger.info("Clustering: %s", name)
    metric_rows = []
    best_k, best_score, best_labels = None, -1, None

    # kmedoids requires float64
    D_f64 = D.astype(np.float64)

    for k in K_RANGE:
        res  = kmedoids.fasterpam(D_f64, k, random_state=42)
        lbls = np.array(res.labels)

        sil = silhouette_score(D_f64, lbls, metric="precomputed")
        db  = davies_bouldin_score(D_f64, lbls)
        ch  = calinski_harabasz_score(D_f64, lbls)

        metric_rows.append({
            "k":                 k,
            "Silhouette":        round(sil, 4),
            "Davies-Bouldin":    round(db,  4),
            "Calinski-Harabasz": round(ch,  2),
            "labels":            lbls,
        })

        if sil > best_score:
            best_score, best_k, best_labels = sil, k, lbls

    results[name] = {
        "best_k":      best_k,
        "best_score":  best_score,
        "best_labels": best_labels,
        "metrics":     pd.DataFrame(metric_rows).drop(columns=["labels"]),
        "all_labels":  {r["k"]: r["labels"] for r in metric_rows},
    }
    print(f"{name:30s} → best k={best_k}, silhouette={best_score:.4f}")

2026-03-17 22:42:52,187 — INFO — Clustering: Mixed (L1+Hamming)
2026-03-17 22:43:00,406 — INFO — Clustering: Mixed (L1+Tanimoto)


Mixed (L1+Hamming)             → best k=2, silhouette=0.2045


2026-03-17 22:43:08,299 — INFO — Clustering: Mixed (L2+Hamming)


Mixed (L1+Tanimoto)            → best k=2, silhouette=0.2004


2026-03-17 22:43:15,732 — INFO — Clustering: Mixed (L2+Tanimoto)


Mixed (L2+Hamming)             → best k=3, silhouette=0.1775


2026-03-17 22:43:22,969 — INFO — Clustering: Mixed (Canberra+Hamming)


Mixed (L2+Tanimoto)            → best k=3, silhouette=0.1832


2026-03-17 22:43:30,224 — INFO — Clustering: Mixed (Canberra+Tanimoto)


Mixed (Canberra+Hamming)       → best k=3, silhouette=0.1557
Mixed (Canberra+Tanimoto)      → best k=3, silhouette=0.1523


In [ ]:
import nbformat
for name, res in results.items():
    mdf = res["metrics"]
    fig = make_subplots(rows=1, cols=3, subplot_titles=[
        "Silhouette (↑ better)",
        "Davies-Bouldin (↓ better)",
        "Calinski-Harabasz (↑ better)",
    ])
    for ci, (col, color) in enumerate([
        ("Silhouette",        "#1f77b4"),
        ("Davies-Bouldin",    "#ff7f0e"),
        ("Calinski-Harabasz", "#2ca02c"),
    ]):
        fig.add_trace(go.Scatter(
            x=mdf["k"].tolist(), y=mdf[col].tolist(),
            mode="lines+markers", line=dict(color=color, width=2),
            marker=dict(size=8), showlegend=False,
        ), row=1, col=ci + 1)
        fig.add_vline(x=res["best_k"], line_dash="dash",
                      line_color="red", row=1, col=ci + 1)

    fig.update_xaxes(title_text="k")
    fig.update_layout(height=380,
                      title_text=f"Validation Metrics — {name} (best k={res['best_k']})")
    fig.show()
    display(mdf)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
umap_embeddings = {}

for name, res in results.items():
    D   = matrices[name]
    lbl = res["best_labels"]
    k   = res["best_k"]

    # 2D
    reducer_2d = umap.UMAP(n_components=2, metric="precomputed",
                            n_neighbors=15, min_dist=0.1, random_state=42)
    emb2d = reducer_2d.fit_transform(D)

    # 3D
    reducer_3d = umap.UMAP(n_components=3, metric="precomputed",
                            n_neighbors=15, min_dist=0.1, random_state=42)
    emb3d = reducer_3d.fit_transform(D)

    umap_embeddings[name] = {"2d": emb2d, "3d": emb3d}

    cluster_labels = [f"Cluster {c}" for c in lbl.tolist()]

    # ── 2D plot ──────────────────────────────────────────────────────────────
    df_2d = pd.DataFrame({
        "UMAP1": emb2d[:, 0], "UMAP2": emb2d[:, 1],
        "Cluster": cluster_labels,
    })
    fig2d = px.scatter(
        df_2d, x="UMAP1", y="UMAP2", color="Cluster",
        title=f"UMAP 2D — {name} (k={k})",
        opacity=0.6,
        color_discrete_sequence=px.colors.qualitative.Vivid,
    )
    fig2d.update_traces(marker=dict(size=4))
    fig2d.update_layout(height=520)
    fig2d.show()

    # ── 3D plot ──────────────────────────────────────────────────────────────
    df_3d = pd.DataFrame({
        "UMAP1": emb3d[:, 0], "UMAP2": emb3d[:, 1], "UMAP3": emb3d[:, 2],
        "Cluster": cluster_labels,
    })
    fig3d = px.scatter_3d(
        df_3d, x="UMAP1", y="UMAP2", z="UMAP3", color="Cluster",
        title=f"UMAP 3D — {name} (k={k})",
        opacity=0.65,
        color_discrete_sequence=px.colors.qualitative.Vivid,
    )
    fig3d.update_traces(marker=dict(size=3))
    fig3d.update_layout(height=620)
    fig3d.show()

In [ ]:
JOB_MAP  = {1: "Unemployed", 2: "Employee", 3: "Manager",
             4: "Entrepreneur", 5: "Retired"}
INV_MAP  = {1: "None", 2: "Lump Sum", 3: "Cap. Accum."}
GEN_MAP  = {0: "Male", 1: "Female"}
AREA_MAP = {1: "North", 2: "Center", 3: "South"}

KEY_FEATURES = ["Income", "Wealth", "Debt", "Saving", "Luxury", "FinEdu"]
PAL          = px.colors.qualitative.Vivid

for name, res in results.items():
    k   = res["best_k"]
    lbl = res["best_labels"]

    df_p = df.copy()
    df_p["Cluster"] = lbl

    num_profile_cols = [c for c in numerical_cols if c in df_p.columns]

    # ── Summary table ─────────────────────────────────────────────────────────
    rows = []
    for c in range(k):
        g   = df_p[df_p["Cluster"] == c]
        row = {"Cluster": c, "N": len(g), "%": f"{len(g)/len(df_p)*100:.1f}%"}
        for col in num_profile_cols:
            row[col] = round(float(g[col].mean()), 3)
        row["Job"]         = JOB_MAP.get(int(g["Job"].mode()[0]), "?")
        row["Gender"]      = GEN_MAP.get(int(g["Gender"].mode()[0]), "?")
        row["Area"]        = AREA_MAP.get(int(g["Area"].mode()[0]), "?")
        rows.append(row)

    print(f"\n{'='*60}")
    print(f" {name} — k={k} cluster summary")
    print(f"{'='*60}")
    display(pd.DataFrame(rows))

    # ── Radar chart ───────────────────────────────────────────────────────────
    radar = go.Figure()
    for c in range(k):
        g    = df_p[df_p["Cluster"] == c]
        vals = [float(g[col].mean()) for col in num_profile_cols]
        radar.add_trace(go.Scatterpolar(
            r=vals + [vals[0]],
            theta=num_profile_cols + [num_profile_cols[0]],
            fill="toself", name=f"Cluster {c}",
            line_color=PAL[c], opacity=0.75,
        ))
    radar.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title=f"Radar — {name} (k={k})",
        height=500,
    )
    radar.show()

    # ── Boxplots for key features ─────────────────────────────────────────────
    box = make_subplots(rows=1, cols=len(KEY_FEATURES),
                        subplot_titles=KEY_FEATURES)
    for ci, col in enumerate(KEY_FEATURES):
        for c in range(k):
            g = df_p[df_p["Cluster"] == c]
            box.add_trace(go.Box(
                y=g[col].tolist(), name=f"Cluster {c}",
                marker_color=PAL[c], showlegend=(ci == 0),
            ), row=1, col=ci + 1)
    box.update_layout(height=420, boxmode="group",
                      title_text=f"Key Features by Cluster — {name}")
    box.show()

In [ ]:
summary_rows = []
for name, res in results.items():
    summary_rows.append({
        "Distance":   name,
        "Best k":     res["best_k"],
        "Silhouette": round(res["best_score"], 4),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("Silhouette", ascending=False)
print("\nFinal comparison — best silhouette per distance metric:")
display(summary_df)

winner = summary_df.iloc[0]["Distance"]
print(f"\nBest performing distance: {winner}")